# Visual Health — fine-tuning partiel RETFound (RFMiD + APTOS)

Fine-tune les 4 derniers blocs de RETFound et la tête sur RFMiD
(`scripts/finetune_retfound.py`, configuration fixée à l'avance), puis
évalue une fois sur le test RFMiD et sur APTOS 2019 (validation externe).
L'analyse (comparaison à la tête linéaire, AUROC à appareil égal sur APTOS)
se fait ensuite en local (`scripts/finetune_analysis.py`).

**Avant de lancer (Colab)** :
1. `Exécution > Modifier le type d'exécution > GPU T4`.
2. Secrets 🔑, accès activé pour ce notebook : `HF_TOKEN` (poids RETFound)
   et `KAGGLE_API_TOKEN` (clé `KGAT_…`, pour APTOS ; règlement de la
   compétition déjà accepté).
3. `Exécution > Tout exécuter` (~45-60 min : préparation des images
   ~15 min, entraînement ~20-30 min, APTOS ~10 min).
4. Le zip `visual_health_finetune.zip` se télécharge à la fin, puis
   **`Exécution > Déconnecter et supprimer l'environnement d'exécution`**
   pour libérer le GPU.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Pas de GPU : choisir GPU T4"

In [ ]:
import os
ON_KAGGLE = os.path.exists("/kaggle/working")
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
%cd {WORK}
![ -d visual-health-platform ] || git clone -q https://github.com/sumarc183-design/visual-health-platform.git
!cd visual-health-platform && git fetch -q && git reset -q --hard origin/main && git log --oneline -1
%cd {WORK}/visual-health-platform
!pip install -q -r requirements.txt kagglehub

In [ ]:
import glob

if ON_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    get_secret = UserSecretsClient().get_secret
else:
    from google.colab import userdata
    get_secret = userdata.get

TARGET = "third_party/retfound/checkpoints/RETFound_mae_natureCFP.pth"
if not os.path.exists(TARGET):
    os.environ["HF_TOKEN"] = get_secret("HF_TOKEN")  # jamais écrit dans le notebook
    !python scripts/download_retfound_checkpoint.py --repo-id YukunZhou/RETFound_mae_natureCFP --out third_party/retfound/checkpoints
CHECKPOINT = sorted(glob.glob("third_party/retfound/checkpoints/*.pth"))[0]
print(CHECKPOINT)

## Données : RFMiD (public) et APTOS (compétition, clé Kaggle)

APTOS : seuls `train.csv` et `train_images/` sont décompressés.

In [ ]:
import kagglehub

RFMID_ROOT = kagglehub.dataset_download("andrewmvd/retinal-disease-classification")
print("RFMiD :", RFMID_ROOT)

APTOS_ROOT = f"{WORK}/aptos"
if not os.path.exists(f"{APTOS_ROOT}/train.csv"):
    token = get_secret("KAGGLE_API_TOKEN").strip()
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    with open(os.path.expanduser("~/.kaggle/access_token"), "w") as f:
        f.write(token)
    os.chmod(os.path.expanduser("~/.kaggle/access_token"), 0o600)
    os.environ["KAGGLE_API_TOKEN"] = token
    !pip install -q -U kaggle
    !kaggle competitions download -c aptos2019-blindness-detection -p {WORK} -q
    !mkdir -p {APTOS_ROOT} && unzip -q -o {WORK}/aptos2019-blindness-detection.zip "train.csv" "train_images/*" -d {APTOS_ROOT}
    !rm {WORK}/aptos2019-blindness-detection.zip
print("APTOS :", APTOS_ROOT, len(os.listdir(f"{APTOS_ROOT}/train_images")), "images")

## Fine-tuning partiel

Configuration fixée à l'avance (4 derniers blocs, 30 epochs au plus, arrêt
après 5 epochs sans progrès en validation). La progression s'affiche epoch
par epoch. Le test RFMiD et APTOS ne sont lus qu'une fois, avec l'epoch
retenue sur la validation.

In [ ]:
import time

t0 = time.time()
!python -u scripts/finetune_retfound.py --data-root "{RFMID_ROOT}" --checkpoint {CHECKPOINT} \
    --out-dir finetune --eval-test --aptos-root "{APTOS_ROOT}" --workers 2 2>&1 | grep -v -i warning
for f in ("finetune/finetune_rfmid.json", "finetune/pred_aptos_finetune.npz"):
    if not os.path.exists(f) or os.path.getmtime(f) < t0:
        raise RuntimeError(f"finetune_retfound.py a échoué : {f} absent (voir la sortie ci-dessus)")

## Garder le résultat

Prédictions et historique (quelques Mo) ; les poids entraînés (~200 Mo)
restent dans la session.

In [ ]:
import zipfile

archive = f"{WORK}/visual_health_finetune.zip"
with zipfile.ZipFile(archive, "w") as z:
    for f in sorted(glob.glob("finetune/*.npz") + glob.glob("finetune/*.json")):
        z.write(f)
print("->", archive, f"({os.path.getsize(archive) / 1e6:.1f} Mo)")
if not ON_KAGGLE:
    from google.colab import files
    files.download(archive)